# Kenya Counties And County-Level Analytics

Kenya has 47 counties, and county governments are responsible for much of day-to-day health service delivery under devolved governance. That makes county-level analytics especially important: trends, quality checks, and forecasts are most useful when they line up with the geography that county health managers actually use for planning and review.

This notebook shows how `khis-toolkit` handles county metadata, county-region groupings, mapping coordinates, and the shift from placeholder IDs to real KHIS organisation-unit IDs once credentials are available.

In [ ]:
import os

import khis

USE_REAL_KHIS = "hiskenya" in os.getenv("DHIS2_BASE_URL", "").lower()
counties_df = khis.list_counties()
counties_df.style.hide(axis="index")

In [ ]:
county_lookup = khis.get_county("Nairobi")
county_lookup

In [ ]:
coast_counties = khis.get_counties_by_region("Coast")
coast_counties

In [ ]:
import folium

county_map = folium.Map(location=[0.2, 37.9], zoom_start=6, tiles="CartoDB positron")
for row in counties_df.itertuples(index=False):
    folium.Marker(
        location=[row.latitude, row.longitude],
        tooltip=row.name,
        popup=f"{row.name} ({row.region})",
    ).add_to(county_map)

county_map

In [ ]:
conn = khis.connect()

if USE_REAL_KHIS:
    df = khis.get(
        conn,
        "malaria",
        counties=["Nairobi", "Kisumu"],
        periods="last_12_months",
    )
    display(df.head())
else:
    preview = {
        "example_call": 'khis.get(conn, "malaria", counties=["Nairobi", "Kisumu"], periods="last_12_months")',
        "resolved_nairobi_id": khis.resolve_org_unit_id("Nairobi"),
        "resolved_kisumu_id": khis.resolve_org_unit_id("Kisumu"),
        "note": "The public DHIS2 demo server does not have Kenya counties, so this direct county pull becomes fully active once you connect real KHIS credentials.",
    }
    preview

In [ ]:
if USE_REAL_KHIS:
    updated = khis.update_from_api(conn)
    {county: updated[county]["dhis2_id"] for county in ["Nairobi", "Mombasa", "Kisumu"]}
else:
    print("When KHIS credentials are available, run: khis.update_from_api(conn)")
    print("That will replace placeholder IDs like KE47 with the real KHIS organisation-unit IDs.")